# **Coding Exercise : #0802_Answer**
Practice: Fine-Tuning(Exercise)

## **0. Import Library**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.metrics import *

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## **1. Load dataset**
Uploading emotions.csv

In [ ]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

In [ ]:
data_path

In [ ]:
data = pd.read_csv(data_path)
data.head()

In [ ]:
sns.countplot(x='label_name', data = data, hue = 'label_name')
plt.grid()
plt.show()

In [ ]:
data['word_count'] = data['text'].apply(lambda x: len(x))
data.head()

In [ ]:
sns.histplot(x='word_count', data = data, bins = 50)
plt.grid()
plt.show()

In [ ]:
labels = data.loc[:, ['label', 'label_name']].drop_duplicates()
labels = labels.sort_values(by = 'label')
print(labels)

label_list = labels['label_name'].tolist()
print(label_list)

## **2. Pre-processing Data**
* save the data with the required columns and split it into train and validation sets.
* convert the data into a tensor dataset structure to be used for training.

In [ ]:
data2 = data.loc[:, ['text','label']].copy()

In [ ]:
# train_test_split
train, val = train_test_split(data2, test_size=0.3, random_state=10)

In [ ]:
# tensor dataset
train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(val)

In [ ]:
train_ds[:3]

## **3. Tokenization**
* download the tokenizer for the model you will use.
* tokenize the dataset’s text column(set max_length = 200)


In [ ]:
# download the tokenizer for the model you will use
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess_function(data):
    return tokenizer(data['text'], padding="max_length", truncation=True, max_length=200)

train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)

## **4. Load Pre-trained Model**
* Now, download the BERT model. At this point, set num_labels = 6 to add an output layer to the BERT model.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 6)

## **5. Set Training Arguments**
* Configure the hyperparameters for model training

In [ ]:
args = TrainingArguments(
    output_dir = './results',          # where to save checkpoints/outputs
    eval_strategy = "epoch",           # run eval at the end of each epoch
    save_strategy = "epoch",           # save a checkpoint every epoch
    learning_rate = 2e-5,              # base learning rate
    per_device_train_batch_size = 32,  # train batch size per device
    per_device_eval_batch_size = 32,   # eval batch size per device
    num_train_epochs  = 3,             # total training epochs
    weight_decay = 0.02,               # L2 weight decay strength
    load_best_model_at_end = True,     # reload best checkpoint (by metric)
    logging_dir ='./logs',             # TensorBoard log directory
    logging_steps = 10,                # log every N training steps
    report_to="tensorboard"            # send logs to TensorBoard
)

## **6. Fine-tune Model**
* Prepare the Trainer to start training the model. You can observe the training process as it runs using the hyperparameters specified earlier.


In [ ]:
trainer = Trainer(
    model = model,                # 학습할 모델
    args = args,                  # TrainingArguments (지정한 하이퍼파라미터)
    train_dataset = train_ds,
    eval_dataset = val_ds,
    tokenizer = tokenizer,
)

In [ ]:
trainer.train()

## **7. Evaluate**
* Make predictions on the validation dataset and evaluate the classification model.
* For evaluating the classification model, we use scikit-learn’s confusion_matrix and classification_report functions.
* To use these functions, the predictions and labels must be converted to NumPy arrays.

In [ ]:
pred_out = trainer.predict(val_ds)
pred = pred_out.predictions.argmax(axis=-1)

In [ ]:
print(confusion_matrix(val_ds['label'], pred))
print(classification_report(val_ds['label'], pred, target_names = label_list))